In [ ]:
# Q1. Embedding a query

# Embed the following query:

#     How does approximate nearest neighbor search work?

# The embedder returns a vector of 384 numbers. What's the first value (v[0])?

In [ ]:

from embedder import Embedder

model = Embedder()

query = "How does approximate nearest neighbor search work?"
v = model.encode(query)

print(v[0])

-0.02058203437252893


In [ ]:
# Q2. Cosine similarity

# The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

# Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

In [10]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [11]:
doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

In [12]:
doc_vector = model.encode(doc["content"])

In [13]:
query = "How does approximate nearest neighbor search work?"
query_vector = model.encode(query)

In [14]:
import numpy as np

similarity = np.dot(query_vector, doc_vector)
print(similarity)

0.36107027225589694


In [ ]:
# Q3. Chunking and search by hand

# A full page covers several topics, which waters down its embedding.

# We chunk the pages the same way as in homework 1:

In [15]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [16]:
texts = [chunk["content"] for chunk in chunks]
X = model.encode_batch(texts)

In [18]:
# Score against query, higher score = more relevant chunk  
scores = X.dot(v)
print(scores)

[ 3.15187611e-01  2.01479664e-01  5.90559037e-02 -7.67661288e-02
  1.18452466e-01 -1.41697012e-01 -2.81406495e-02 -4.65669117e-02
 -2.06994543e-02 -6.07744147e-02  2.13273769e-01  8.87600958e-02
 -1.97269268e-02  3.11629985e-01  5.51034635e-01  2.04008152e-01
  2.12515802e-01  1.93649107e-01  2.51961267e-01  1.31078610e-01
  2.59120607e-01  7.63816369e-02  9.59193203e-02  9.81471228e-03
 -3.59107168e-02  1.01211406e-02  1.10786940e-01 -9.90259915e-02
 -3.71170645e-02  7.59057333e-02 -3.35340234e-02  8.86841484e-03
  1.02636448e-01  6.89615413e-02  1.29408854e-01  2.57709121e-01
  3.23680576e-01  1.06350076e-01  5.61891208e-02  2.34017441e-01
  1.97954358e-01  9.64296342e-02  1.93709934e-01  2.16719269e-01
  3.48340450e-01  5.10906541e-02  2.05212833e-01  1.05416182e-01
 -3.25432660e-02  4.94665347e-02  2.38574873e-01 -3.44207304e-02
  1.82165430e-01  2.77929421e-02 -2.26806900e-03  2.91897549e-01
  4.39332202e-02  2.23759662e-01  1.63523531e-01  9.16125960e-02
 -8.39985178e-02  7.88591

In [19]:
# Find best chunk
best_idx = scores.argmax()
chunks[best_idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'